# 10 — KMeans + silhouette (+ optional HDBSCAN)

**Data:** PPI (or other) network as an undirected edgelist, embedded with **native 3D hyperbolic D-Mercator** ([`networkgeometry/d-mercator`](https://github.com/networkgeometry/d-mercator)) → `edges_GC.inf_coord` + `edges_GC.edge`. This notebook reads the merged table from `cache/merged.parquet` (see `00_load_and_sanity.ipynb`).

**Dependencies:** `pip install scikit-learn pandas pyarrow matplotlib` and, for density clustering, `pip install hdbscan`.


### What this notebook does (and what it does *not* do)

**Feature matrix (same as before):** for every protein/node row, concatenate

1. **Unit direction** — all `Inf.Pos.*` columns from D-Mercator, L2-normalized to the sphere \(S^{D}\) (here \(D=3\) → four coordinates). This captures *angular* similarity in hyperbolic space.
2. **`log1p(Inf.Hyp.Rad)`** — one radial coordinate after a monotone transform. Radial layers in hyperbolic embeddings often correlate with popularity / degree structure; see your UMAP notebooks for geometry intuition.

All clustering is in **Euclidean space on this concatenated vector**. That is *not* the same as clustering by true hyperbolic geodesic distance (your `native_hyperbolic.py` path); it is a practical, cheap probe of “blobs” in the exported coordinates.

**KMeans (partitioning):** assigns every point to exactly one of \(k\) clusters. **Silhouette** (higher is better) summarizes how similar a point is to its own cluster versus others, in the chosen metric (default Euclidean on `X`). **Davies–Bouldin** (lower is better) is another internal index that does not need a \(k\)-sweep interpretability legend—use it as a cross-check.

**HDBSCAN (density / hierarchical):** finds dense regions and can label points as **noise** (`-1`). Tuning **`min_cluster_size`** (minimum points to form a seed cluster) and **`min_samples`** (how conservative the density estimate is) trades **number of clusters**, **noise fraction**, and **minimum meaningful module size**. There is no single “correct” setting: biologically you may *want* many small tight groups (smaller `min_cluster_size`, often more noise) or a few large basins (larger `min_cluster_size`, often less noise).

**Full graph vs subsampling:** Clustering and HDBSCAN run on **all rows** in `merged.parquet` (no fixed 60k cap). **Silhouette** on millions of points is prohibitively expensive (pairwise structure); when \(N\) is large we still **fit KMeans on the full `X`**, but evaluate silhouette on a **large random subset** (`SILHOUETTE_SAMPLE_SIZE`) so the curve remains statistically stable. Adjust that constant if you need tighter estimates.

**Interpretation:** Peaks in internal metrics suggest *geometry in this feature space*, not validated functional modules. Cross-check with `07–09` UMAP views, `03` link prediction, or external annotations if you need biological claims.


In [ ]:
from __future__ import annotations

import time
import warnings
from itertools import product
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import davies_bouldin_score, silhouette_score

import dmercator3d_io as dm

# --- reproducibility & ranges -------------------------------------------------
RNG_SEED = 5
rng = np.random.default_rng(RNG_SEED)
KMEANS_K_GRID = range(2, 9)
KMEANS_N_INIT = 10

# Silhouette is expensive on huge N; we still cluster on ALL rows, but score on a subset.
# Set to None to use every row (only feasible for modest N, e.g. < ~30k).
SILHOUETTE_SAMPLE_SIZE: int | None = 60_000

# HDBSCAN tuning grid (edit freely). Larger grids × large N can take a long time.
HDBSCAN_MIN_CLUSTER_SIZES = (30, 50, 100, 200, 400)
HDBSCAN_MIN_SAMPLES_LIST = (None, 5, 15, 40)  # None → library default (usually equals mcs)

warnings.filterwarnings(
    "ignore",
    message=".*force_all_finite.*",
    category=FutureWarning,
)


def _silhouette_maybe_subsample(
    X: np.ndarray, labels: np.ndarray, sample_size: int | None, random_state: int
) -> float:
    n = X.shape[0]
    if sample_size is None or n <= sample_size:
        return float(silhouette_score(X, labels, random_state=random_state))
    return float(
        silhouette_score(
            X, labels, sample_size=sample_size, random_state=random_state
        )
    )


def _cluster_size_summary(labels: np.ndarray) -> pd.Series:
    s = pd.Series(labels)
    vc = s.value_counts()
    if -1 in vc.index:
        noise = int(vc.loc[-1])
        vc_body = vc.drop(index=-1, errors="ignore")
    else:
        noise = 0
        vc_body = vc
    sizes = vc_body.to_numpy(dtype=np.int64)
    return pd.Series(
        {
            "n_noise": noise,
            "noise_frac": noise / len(labels),
            "n_clusters": int(vc_body.shape[0]),
            "smallest_cluster_size": int(sizes.min()) if len(sizes) else np.nan,
            "largest_cluster_size": int(sizes.max()) if len(sizes) else np.nan,
            "median_cluster_size": float(np.median(sizes)) if len(sizes) else np.nan,
        }
    )


print("CONFIG loaded.")
print("  KMeans k grid:", list(KMEANS_K_GRID))
print("  Silhouette sample_size:", SILHOUETTE_SAMPLE_SIZE)
print("  HDBSCAN mcs:", HDBSCAN_MIN_CLUSTER_SIZES, " min_samples:", HDBSCAN_MIN_SAMPLES_LIST)


### Load merged table and build `X`

`merged.parquet` is expected to contain `Inf.Hyp.Rad` and all `Inf.Pos.*` from the D-Mercator `*.inf_coord` export (see [`dmercator3d_io.py`](dmercator3d_io.py)). Rows are **vertices / proteins**; the original PPI is the undirected edgelist D-Mercator ingested ([upstream tool](https://github.com/networkgeometry/d-mercator)).

In [ ]:
t0 = time.perf_counter()
merged = dm.load_merged_parquet(Path("cache/merged.parquet"))
load_s = time.perf_counter() - t0

print("=== merged.parquet ===")
print(f"  rows (vertices): {len(merged):,}")
print(f"  columns ({len(merged.columns)}): {list(merged.columns)[:12]}{' ...' if len(merged.columns) > 12 else ''}")
print(f"  load time: {load_s:.2f}s")

# Optional: echo inferred dimension / run paths (helps provenance in saved HTML exports)
try:
    p = dm.paths_for_run()
    meta, _ = dm.parse_inf_coord(p["inf_coord"])
    print("\n=== D-Mercator run metadata (from inf_coord header) ===")
    for key in ("dimension", "n_pos", "nb_vertices", "beta", "mu", "inf_coord_path"):
        if key in meta:
            print(f"  {key}: {meta[key]}")
except Exception as exc:  # noqa: BLE001 — notebook UX
    print("\n(inf_coord metadata not printed:", exc, ")")

if "Inf.Hyp.Rad" not in merged.columns:
    raise KeyError("merged parquet must contain Inf.Hyp.Rad")

rad = merged["Inf.Hyp.Rad"].to_numpy(dtype=np.float64)
print("\n=== Inf.Hyp.Rad (raw) ===")
print(pd.Series(rad).describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).to_string())

t1 = time.perf_counter()
U = dm.normalize_direction_nd(merged)
log_rad = np.log1p(rad)[:, None]
X = np.hstack([U, log_rad]).astype(np.float64, copy=False)
feat_s = time.perf_counter() - t1

n, d_feat = X.shape
print("\n=== Feature matrix X ===")
print(f"  shape: {X.shape}  (direction + log1p radius)")
print(f"  build time: {feat_s:.2f}s")
print(f"  dtype: {X.dtype}  nbytes ~ {X.nbytes / 1e9:.3f} GB")

bad = ~np.isfinite(X).all(axis=1)
if bad.any():
    print(f"  WARNING: {bad.sum():,} rows with non-finite X — dropping for clustering")
    merged = merged.loc[~bad].reset_index(drop=True)
    X = X[~bad]
    n = X.shape[0]

# Quick per-column scale (helps spot if radius dominates numerically)
col_std = X.std(axis=0)
print("\n=== per-dimension std (after assembly) ===")
print("  Inf.Pos.* (unit) std ~", float(np.mean(col_std[:-1])), " | log1p(rad) std =", float(col_std[-1]))


### KMeans on **full** `X`

We record **inertia** (within-cluster sum of squares), **Davies–Bouldin** on the full data, and **silhouette** (optionally subsampled for scoring only — see `SILHOUETTE_SAMPLE_SIZE` above).

In [ ]:
rows = []
labels_by_k: dict[int, np.ndarray] = {}

print("=== KMeans sweep (full N) ===")
for k in KMEANS_K_GRID:
    t0 = time.perf_counter()
    km = KMeans(
        n_clusters=k,
        random_state=RNG_SEED,
        n_init=KMEANS_N_INIT,
    )
    lab = km.fit_predict(X)
    fit_s = time.perf_counter() - t0
    labels_by_k[k] = lab

    t1 = time.perf_counter()
    sil = _silhouette_maybe_subsample(X, lab, SILHOUETTE_SAMPLE_SIZE, RNG_SEED)
    db = float(davies_bouldin_score(X, lab))
    db_s = time.perf_counter() - t1

    vc = pd.Series(lab).value_counts()
    rows.append(
        {
            "k": k,
            "inertia": float(km.inertia_),
            "silhouette": sil,
            "davies_bouldin": db,
            "fit_s": fit_s,
            "metrics_s": db_s,
            "smallest_cluster": int(vc.min()),
            "largest_cluster": int(vc.max()),
        }
    )
    print(
        f"  k={k:2d}  silhouette={sil:.4f}  DB={db:.4f}  inertia={km.inertia_:.3g}  "
        f"cluster_size[{vc.min()}, {vc.max()}]  fit={fit_s:.2f}s metrics={db_s:.2f}s"
    )

kmeans_df = pd.DataFrame(rows).set_index("k")
print("\nSummary table:")
print(kmeans_df.to_string())

best_k_sil = int(kmeans_df["silhouette"].idxmax())
best_k_db = int(kmeans_df["davies_bouldin"].idxmin())
print(
    f"\nHeuristic picks:  max silhouette → k={best_k_sil} ; "
    f"min Davies–Bouldin → k={best_k_db}"
)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11, 3.2))

axes[0].plot(kmeans_df.index, kmeans_df["silhouette"], marker="o", color="C0")
axes[0].set_xlabel("k")
axes[0].set_ylabel("silhouette")
axes[0].set_title(
    "Silhouette (subsampled score)"
    if SILHOUETTE_SAMPLE_SIZE is not None and n > SILHOUETTE_SAMPLE_SIZE
    else "Silhouette (full N)"
)

axes[1].plot(kmeans_df.index, kmeans_df["davies_bouldin"], marker="o", color="C1")
axes[1].set_xlabel("k")
axes[1].set_ylabel("Davies–Bouldin (lower better)")
axes[1].set_title("Davies–Bouldin (full N)")

axes[2].plot(kmeans_df.index, kmeans_df["inertia"], marker="o", color="C2")
axes[2].set_xlabel("k")
axes[2].set_ylabel("inertia")
axes[2].set_title("KMeans inertia (full N)")

fig.suptitle("KMeans internal metrics — direction + log1p(Inf.Hyp.Rad)", y=1.02)
plt.tight_layout()
plt.show()

# Cluster size distribution for a couple of interesting k values
for kk in sorted({best_k_sil, best_k_db, 2, 4} & set(labels_by_k.keys())):
    lab = labels_by_k[kk]
    vc = pd.Series(lab).value_counts().sort_index()
    print(f"\nKMeans k={kk} cluster counts (sorted by cluster id):\n{vc.to_string()}")


### HDBSCAN grid (density hierarchy)

We sweep **`min_cluster_size`** and **`min_samples`** on the **same full `X`**. Each row reports how many **density clusters** emerge and what fraction of proteins stay in the **noise** bucket (`label == -1`). Use the table to pick a point on the **resolution / noise** trade-off that matches how you plan to use the groups (e.g. hypothesis generation vs strict module calling).

Requires `pip install hdbscan`. If the import fails, skip this section.

In [ ]:
try:
    import hdbscan  # type: ignore[import-not-found]
except ImportError:
    hdbscan = None
    print("hdbscan not installed — skip HDBSCAN section (`pip install hdbscan`).")

hdb_rows: list[dict] = []
hdb_labels_ref: np.ndarray | None = None
hdb_ref_key: tuple | None = None

if hdbscan is not None:
    n = X.shape[0]
    if n > 150_000:
        print(
            f"NOTE: HDBSCAN on N={n:,} can be slow / memory-heavy. "
            "Consider narrowing the grid or setting a machine-specific cap."
        )

    print("=== HDBSCAN parameter grid (full N) ===")
    label_cache: dict[tuple[int, int | str], np.ndarray] = {}
    for mcs, ms in product(HDBSCAN_MIN_CLUSTER_SIZES, HDBSCAN_MIN_SAMPLES_LIST):
        kwargs = {"min_cluster_size": int(mcs)}
        if ms is not None:
            kwargs["min_samples"] = int(ms)
        t0 = time.perf_counter()
        cl = hdbscan.HDBSCAN(**kwargs).fit(X)
        lab = cl.labels_
        cache_key = (int(mcs), int(ms) if ms is not None else "default")
        label_cache[cache_key] = lab.copy()
        fit_s = time.perf_counter() - t0
        summ = _cluster_size_summary(lab)
        row = {
            "min_cluster_size": mcs,
            "min_samples": ms if ms is not None else "default",
            "n_clusters": int(summ["n_clusters"]),
            "n_noise": int(summ["n_noise"]),
            "noise_frac": float(summ["noise_frac"]),
            "smallest_cluster_size": summ["smallest_cluster_size"],
            "largest_cluster_size": summ["largest_cluster_size"],
            "median_cluster_size": summ["median_cluster_size"],
            "fit_s": fit_s,
            "prob_missing_frac": float(np.isnan(cl.probabilities_).mean()),
        }
        hdb_rows.append(row)
        ms_disp = ms if ms is not None else "def"
        print(
            f"  mcs={mcs:4d}  min_samples={str(ms_disp):>4s}  clusters={row['n_clusters']:4d}  "
            f"noise={row['n_noise']:7,} ({row['noise_frac']*100:5.1f}%)  "
            f"size[{summ['smallest_cluster_size']:.0f},{summ['largest_cluster_size']:.0f}]  "
            f"t={fit_s:.2f}s"
        )

    hdb_df = pd.DataFrame(hdb_rows)
    print("\nFull grid (sorted by noise_frac, then n_clusters):")
    print(
        hdb_df.sort_values(["noise_frac", "n_clusters"], ascending=[True, False]).to_string(
            index=False
        )
    )

    # Heuristic "display" fit: among configs with at least 3 clusters and <=35% noise, maximize n_clusters.
    cand = hdb_df[(hdb_df["n_clusters"] >= 3) & (hdb_df["noise_frac"] <= 0.35)].copy()
    if len(cand) == 0:
        cand = hdb_df.sort_values("noise_frac").head(5)
        print(
            "\n(no config met n_clusters>=3 & noise<=35%; showing 5 lowest-noise rows for reference)"
        )
    pick = cand.sort_values(["n_clusters", "noise_frac"], ascending=[False, True]).iloc[0]
    mcs_pick = int(pick["min_cluster_size"])
    ms_pick = pick["min_samples"]
    print(
        f"\nHeuristic pick for downstream plots: min_cluster_size={mcs_pick}, "
        f"min_samples={ms_pick}  (n_clusters={int(pick['n_clusters'])}, "
        f"noise_frac={pick['noise_frac']*100:.1f}%)"
    )

    ms_key: int | str = int(ms_pick) if ms_pick != "default" else "default"
    hdb_labels_ref = label_cache[(mcs_pick, ms_key)].copy()
    hdb_ref_key = (mcs_pick, ms_pick)
    print("  (reusing labels from grid — no extra HDBSCAN fit)")


In [ ]:
if hdbscan is not None and hdb_labels_ref is not None:
    lab = hdb_labels_ref
    vc = pd.Series(lab).value_counts().sort_index()
    noise_n = int((lab == -1).sum())
    print("=== HDBSCAN label histogram (heuristic pick) ===")
    print(f"  key: min_cluster_size / min_samples = {hdb_ref_key}")
    print(f"  noise -1: {noise_n:,} ({noise_n / len(lab) * 100:.2f}%)")
    body = vc[vc.index != -1]
    print(f"  labelled clusters: {body.shape[0]}")
    print("\nTop 15 largest clusters (by count):")
    print(body.sort_values(ascending=False).head(15).to_string())

    fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))

    sizes = body.sort_values(ascending=False).to_numpy()
    axes[0].bar(range(len(sizes)), sizes, color="C3")
    axes[0].set_yscale("log")
    axes[0].set_xlabel("cluster rank (by size)")
    axes[0].set_ylabel("size (log scale)")
    axes[0].set_title("HDBSCAN cluster sizes (excl. noise)")

    # Noise vs mcs for each min_samples level (line plot)
    for ms in hdb_df["min_samples"].unique():
        sub = hdb_df[hdb_df["min_samples"] == ms].sort_values("min_cluster_size")
        axes[1].plot(
            sub["min_cluster_size"],
            sub["noise_frac"] * 100.0,
            marker="o",
            label=f"min_samples={ms}",
        )
    axes[1].set_xlabel("min_cluster_size")
    axes[1].set_ylabel("noise %")
    axes[1].set_title("Grid: noise vs min_cluster_size")
    axes[1].legend(fontsize=8, loc="best")
    plt.tight_layout()
    plt.show()
else:
    print("HDBSCAN plots skipped.")


### Agreement between KMeans and HDBSCAN (sanity check)

**Adjusted Rand index (ARI)** compares two clusterings on the *same points* (chance-corrected, symmetric in its arguments). It is **not** external biological validation. We report ARI on a random subset for speed.

Interpreting alongside **noise**: HDBSCAN’s `-1` labels are not “a cluster”; comparing KMeans to HDBSCAN on **all** points treats noise as its own label bucket. The **masked** variant below restricts to proteins that received a **proper HDBSCAN cluster id**, which is usually the fairer comparison to “coarse KMeans regions vs dense cores”.

In [ ]:
from sklearn.metrics import adjusted_rand_score

ari_subset = min(50_000, X.shape[0])
ix = rng.choice(X.shape[0], size=ari_subset, replace=False)
lab_km = labels_by_k[best_k_sil][ix]

print("=== KMeans vs HDBSCAN (Adjusted Rand) ===")
print(f"  random subset size: {ari_subset:,}  (KMeans labels: k={best_k_sil})")

if hdbscan is None or hdb_labels_ref is None:
    print("  (HDBSCAN not available — skip)")
else:
    lab_h = hdb_labels_ref[ix]
    ari_all = adjusted_rand_score(lab_km, lab_h)
    print(f"  ARI on all subset points (noise = label -1): {ari_all:.4f}")

    mask = lab_h != -1
    n_core = int(mask.sum())
    if n_core > 200:
        ari_core = adjusted_rand_score(lab_km[mask], lab_h[mask])
        print(
            f"  ARI restricted to HDBSCAN non-noise points in subset: {ari_core:.4f}  "
            f"(n_core={n_core:,} / {ari_subset:,})"
        )
    else:
        print("  (too few non-noise points in subset for masked ARI)")

    # Quick size overlap: how pure are KMeans blobs inside each HDBSCAN cluster?
    sub_idx = ix[mask]
    if n_core > 200:
        km_full = labels_by_k[best_k_sil][sub_idx]
        h_full = hdb_labels_ref[sub_idx]
        ct = pd.crosstab(
            pd.Series(km_full, name="KMeans"),
            pd.Series(h_full, name="HDBSCAN"),
        )
        if ct.size:
            print("\n  Crosstab (rows=KMeans, cols=HDBSCAN) on masked subset — top block:")
            with pd.option_context("display.max_rows", 12, "display.max_columns", 12):
                print(ct.iloc[: min(12, ct.shape[0]), : min(12, ct.shape[1])])
        else:
            print("\n  (empty crosstab — degenerate clustering on subset)")


### Optional: side-by-side philosophy

| Method | Every protein gets a cluster? | What “good” looks like here |
|--------|--------------------------------|------------------------------|
| **KMeans** | Yes | Low DB + silhouette peak at moderate \(k\) — a coarse global partition of the feature cloud. |
| **HDBSCAN** | No (noise allowed) | You accept some `-1` labels in exchange for sharper, density-defined groups; tune `min_cluster_size` / `min_samples` until the **noise vs cluster count** curve matches your scientific tolerance. |

Neither replaces **graph** community detection on the PPI itself (Louvain / etc. in other notebooks): these clusters live in **D-Mercator’s inferred geometry** ([tool](https://github.com/networkgeometry/d-mercator)), which is already a compressed summary of the edgelist.